# Milestone 3 · Optimiser les livraisons fournisseur × usine

**49 camions au jour de pointe, 759 sur la période, 65 106,34 € d’économies calculées.**

Sur 4 389 lignes de besoin, on propose plusieurs calendriers par couple, puis on les choisit ensemble pour éviter les pointes. La recherche approfondie porte sur les 80 couples les plus coûteux.

La démonstration lit les résultats publiés. [Présentation PDF](../milestone2/planification_testbed/presentation-ensemble.pdf) · [Données et exports](../milestone2/planification_testbed/ensemble-cible/)

In [1]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd().resolve()
root = next(p for p in (cwd, *cwd.parents) if (p / 'milestone2/planification_testbed').is_dir())
data = root / 'milestone2/planification_testbed/ensemble-cible'
comparatif = pd.read_csv(data / 'comparatif.csv').set_index('scenario')
mesures = pd.read_csv(data / 'mesures.csv').set_index('mode')

scenarios = {
    'main': 'Main de référence',
    'strict_budget_main_pic_global': 'RG06 sous budget',
    'rg05_rg06_avant_ciblage': 'RG05 + RG06, avant ciblage',
    'rg05_rg06_moins_cher': 'RG05 + RG06, après ciblage',
}
colonnes = ['cost_eur', 'trucks', 'peak']
table = comparatif.loc[list(scenarios), colonnes].rename(index=scenarios)
print(table.rename(columns={'cost_eur': 'Coût (€)', 'trucks': 'Camions', 'peak': 'Pic/jour'}).to_string())
gain = mesures.loc['80 couples', 'reference_eur'] - table.iloc[-1]['cost_eur']
print(f'\nÉconomies calculées par rapport à la référence reconstruite : {gain:.2f} €')

                             Coût (€)  Camions  Pic/jour
scenario                                                
Main de référence           110329.59      761        41
RG06 sous budget            110314.80      733        32
RG05 + RG06, avant ciblage  111618.27      759        52
RG05 + RG06, après ciblage  111618.27      759        49

Économies calculées par rapport à la référence reconstruite : 65106.34 €


## 1. Construire un calendrier sous RG06

Chaque ligne peut partir de **J−7 à J**, un jour ouvré allemand. Avec un départ lundi puis jeudi, le lundi couvre les besoins de lundi à mercredi inclus. Les besoins de jeudi partent jeudi.

Dans le graphe de Wissam, **une date possible est un nœud**. Une arête relie deux départs successifs. Son prix est le coût des camions qui couvrent les besoins entre ces deux départs. Le plus court chemin choisit la suite de départs qui minimise la somme de ces coûts. La qualité de ce résultat dépend du prix donné aux arêtes par le chargeur glouton.

**RG05** demande des jours fixes et au moins une semaine qui utilise tous les jours du profil. Les autres semaines peuvent utiliser moins de jours.

In [2]:
jours = ['lundi', 'mardi', 'mercredi', 'jeudi', 'vendredi']
departs = (0, 3)
for besoin in range(len(jours)):
    depart = max(j for j in departs if j <= besoin)
    print(f'Besoin {jours[besoin]:9s} → départ {jours[depart]}')

Besoin lundi     → départ lundi
Besoin mardi     → départ lundi
Besoin mercredi  → départ lundi
Besoin jeudi     → départ jeudi
Besoin vendredi  → départ jeudi


## 2. Choisir les calendriers ensemble

On reprend les calendriers de main, de notre heuristique et de Wissam. Le graphe génère des candidats supplémentaires, dont des calendriers qui évitent les trois dates les plus chargées. On recharge toutes les propositions de la même façon : lignes entières d’un seul couple par camion, **24 500 kg et 110,4 m³ au plus**, puis un prix par camion selon la distance et le poids chargé.

On garde les candidats qui passent RG05 et RG06. Le solveur prend **un calendrier par couple**, additionne les camions jour par jour et minimise le pic global sous un plafond de coût. Dans le vivier testé, le plafond de main est trop bas pour satisfaire les deux règles. On retient alors le coût minimal compatible, puis le plus petit pic à ce coût.

Dans l’exemple illustratif ci-dessous, le meilleur choix local peut envoyer les deux camions le lundi. Un choix coordonné en place un le jeudi au même coût.

In [3]:
from itertools import product
from collections import Counter

options_a = [('lundi', 100), ('jeudi', 100)]
options_b = [('lundi', 120), ('jeudi', 125)]
budget = 220
possibles = []
for choix in product(options_a, options_b):
    cout = sum(prix for _, prix in choix)
    if cout <= budget:
        camions_par_jour = Counter(jour for jour, _ in choix)
        possibles.append((max(camions_par_jour.values()), cout, choix))

pic, cout, choix = min(possibles)
print(f'Pic : {pic} camion par jour | Coût : {cout} € | Départs : {choix}')

Pic : 1 camion par jour | Coût : 220 € | Départs : (('jeudi', 100), ('lundi', 120))


## 3. Voir ce que le ciblage change

**De 52 à 49 au pic, au même coût**, sur le même vivier à 80 couples. Le plan à 32 respecte RG06, mais 56 couples n’atteignent jamais leur profil complet dans une semaine.

![Carte de chaleur des camions par jour, échelle commune](../milestone2/planification_testbed/heatmap-ensemble-cible.svg)

Le plan à 49 dépasse de **1 288,68 €** le plafond expérimental de **110 329,59 €**, tiré du main de référence (c069c90). Les **85,03 s** mesurées comprennent la régénération de l’heuristique et les exports, en réutilisant les calendriers publiés de main et de Wissam.

## 4. Vérifier le plan livré

Les deux cellules suivantes relisent les CSV. Elles contrôlent les affectations, les charges, le délai, le dernier départ et les profils. Les jours fériés nationaux allemands sont pris en compte par le moteur. Les exports signalent les poids, volumes ou distances imputés en amont. La préférence de fréquence RG07 reste un contrôle distinct.

In [4]:
aff = pd.read_csv(data / 'affectations.csv', parse_dates=['date', 'due'])
camions = pd.read_csv(data / 'camions.csv', parse_dates=['date'])
couples = ['supplier', 'plant']

assert len(aff) == aff['id'].nunique() == 4389
assert (aff['due'] - aff['date']).dt.days.between(0, 7).all()
assert aff['date'].dt.dayofweek.lt(5).all()
assert aff.groupby('truck_id')[couples].nunique().le(1).all().all()
charges = aff.groupby('truck_id')[['kg', 'm3']].sum().sort_index()
pd.testing.assert_frame_equal(charges, camions.set_index('truck_id')[['kg', 'm3']].sort_index())
assert charges['kg'].le(24_500 + 1e-8).all()
assert charges['m3'].le(110.4 + 1e-8).all()
assert len(camions) == 759 and camions.groupby('date').size().max() == 49
print('4 389 lignes uniques, dates et capacités vérifiées. Pic : 49 camions.')

4 389 lignes uniques, dates et capacités vérifiées. Pic : 49 camions.


In [5]:
jours = aff[couples + ['date']].drop_duplicates().sort_values('date')
jours['suivant'] = jours.groupby(couples)['date'].shift(-1)
controle = aff.merge(jours, on=couples + ['date'], validate='m:1')
assert not controle['due'].ge(controle['suivant']).any()  # RG06

jours['jour'] = jours['date'].dt.dayofweek
jours['semaine'] = jours['date'] - pd.to_timedelta(jours['jour'], unit='D')
jours_fixes = jours.groupby(couples)['jour'].nunique()
semaine_max = jours.groupby(couples + ['semaine']).size().groupby(level=[0, 1]).max()
assert jours_fixes.eq(semaine_max).all()  # RG05
print(f'RG05 et RG06 vérifiées sur {len(jours_fixes)} couples.\n')
print(pd.read_csv(data / 'economies-par-usine.csv').to_string(index=False))

RG05 et RG06 vérifiées sur 180 couples.

  plant  trucks  baseline_eur  plan_eur  saving_eur
PLT_001     218      45200.68  28290.73    16909.95
PLT_002     220      45004.43  29126.66    15877.77
PLT_003     108      27931.81  19590.46     8341.36
PLT_004     213      58587.68  34610.42    23977.26


## 5. Recalcul complet, si nécessaire

La démonstration ci-dessus ne requiert que pandas. Pour refaire le calcul, installer aussi NumPy, SciPy et holidays, puis récupérer les branches main et milestone3-candidat-wissam avec git fetch origin. Passer RECALCULER à True pour appeler le moteur. Les exports du nouveau run vont dans results/milestone3-recalcul/.

In [6]:
RECALCULER = False
if RECALCULER:
    import os, subprocess, sys

    script = root / 'milestone2/planification_testbed/ensemble_calendriers.py'
    env = {**os.environ, 'PYTHONPATH': str(root / 'src')}
    subprocess.run([sys.executable, str(script), '--repo', str(root),
                    '--output', 'results/milestone3-recalcul',
                    '--enrich-peaks', '--profile-pairs', '80'],
                   cwd=root, env=env, check=True)